# Package 09 — High-speed deploy: ONNX export, consistency, latency

**Author:** Serhii Polishchuk  
**Course:** Computer Vision Engineering for DefenceTech  
**Weights:** `yolov8n.pt` (same head as Report 05)  
**imgsz:** 640  

Steps: Setup → Export ONNX → Consistency (PyTorch vs ONNX) → Latency table → Notes on TensorRT.

## 1. Setup

In [ ]:
!pip install -q ultralytics onnx onnxruntime opencv-python-headless
import time, json, platform
from pathlib import Path
import numpy as np
import cv2
import torch
from ultralytics import YOLO

OUT = Path("/content/pkg09_out")
OUT.mkdir(parents=True, exist_ok=True)
IMGSZ = 640
CONF = 0.25
IOU = 0.7
OPSET = 12
WEIGHTS = "yolov8n.pt"

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU", torch.cuda.get_device_name(0))
print("platform", platform.platform())

## 2. Export ONNX

Static export, imgsz=640 as Report 05. NMS left to postprocess in runtime comparison (Ultralytics predict path).

In [ ]:
model = YOLO(WEIGHTS)
# Ultralytics export
export_path = model.export(format="onnx", imgsz=IMGSZ, opset=OPSET, simplify=True, dynamic=False)
print("exported:", export_path)

onnx_src = Path(export_path)
onnx_dst = OUT / "yolov8n_imgsz640_opset12.onnx"
if onnx_src.resolve() != onnx_dst.resolve():
    import shutil
    shutil.copy(onnx_src, onnx_dst)
print("saved", onnx_dst, "size_MB", round(onnx_dst.stat().st_size/1e6, 2))

(OUT / "export_meta.json").write_text(json.dumps({
    "weights": WEIGHTS,
    "imgsz": IMGSZ,
    "opset": OPSET,
    "dynamic": False,
    "simplify": True,
    "command": f"YOLO('{WEIGHTS}').export(format='onnx', imgsz={IMGSZ}, opset={OPSET}, simplify=True, dynamic=False)",
}, indent=2))
print("meta ok")

## 3. Control frames (from VisDrone val or synthetic)

If VisDrone not mounted, download a few public samples / use COCO8 built into ultralytics for consistency + latency protocol.

In [ ]:
# Prefer VisDrone val if present from earlier DZs; else ultralytics assets
vd = Path("/content/datasets/VisDrone/VisDrone2019-DET-val/images")
if not vd.exists():
    vd = Path("/content/tum")  # unlikely images
frames = sorted(vd.glob("*.jpg"))[:30] if vd.exists() else []

if len(frames) < 5:
    # fallback: YOLO downloads coco8
    from ultralytics.utils.downloads import safe_download
    import zipfile
    z = Path("/content/coco8.zip")
    if not Path("/content/coco8").exists():
        safe_download("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip", z)
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content")
    frames = sorted(Path("/content/coco8/images/train").glob("*.jpg"))[:8]
    # duplicate to have more timing samples
    frames = (frames * 8)[:32]

print("n frames", len(frames), "example", frames[0] if frames else None)
CTRL = frames[:16]  # fixed control set for consistency
BENCH = frames[:32]

## 4. Consistency: framework (PyTorch Ultralytics) vs ONNXRuntime

Match boxes by class + IoU ≥ 0.5. Record score delta for matched pairs.

**Thresholds (fixed before run):** `iou_match=0.5`, `conf=0.25`.

In [ ]:
import onnxruntime as ort

IOU_MATCH = 0.5
SCORE_NEAR = 0.05  # note unmatched near conf threshold

def xyxy_iou(a, b):
    # a,b: (4,)
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    uni = area_a + area_b - inter + 1e-9
    return inter / uni

def preds_torch(model, img_path):
    r = model.predict(str(img_path), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)[0]
    if r.boxes is None or len(r.boxes)==0:
        return []
    xyxy = r.boxes.xyxy.cpu().numpy()
    conf = r.boxes.conf.cpu().numpy()
    cls = r.boxes.cls.cpu().numpy().astype(int)
    return [{"xyxy": xyxy[i], "conf": float(conf[i]), "cls": int(cls[i])} for i in range(len(cls))]

# ONNX via ultralytics YOLO(onnx) — same postprocess head as export
model_onnx = YOLO(str(onnx_dst))

def preds_onnx(img_path):
    r = model_onnx.predict(str(img_path), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)[0]
    if r.boxes is None or len(r.boxes)==0:
        return []
    xyxy = r.boxes.xyxy.cpu().numpy()
    conf = r.boxes.conf.cpu().numpy()
    cls = r.boxes.cls.cpu().numpy().astype(int)
    return [{"xyxy": xyxy[i], "conf": float(conf[i]), "cls": int(cls[i])} for i in range(len(cls))]

matched_scores = []
n_match = 0
n_pt = 0
n_ox = 0
unmatched_pt = 0
unmatched_ox = 0
near_thr_unmatched = 0

for p in CTRL:
    A = preds_torch(model, p)
    B = preds_onnx(p)
    n_pt += len(A); n_ox += len(B)
    used = set()
    for a in A:
        best_j, best_iou = None, 0
        for j, b in enumerate(B):
            if j in used or b["cls"] != a["cls"]:
                continue
            iou = xyxy_iou(a["xyxy"], b["xyxy"])
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_j is not None and best_iou >= IOU_MATCH:
            used.add(best_j)
            n_match += 1
            matched_scores.append(abs(a["conf"] - B[best_j]["conf"]))
        else:
            unmatched_pt += 1
            if a["conf"] < CONF + SCORE_NEAR:
                near_thr_unmatched += 1
    unmatched_ox += (len(B) - len(used))

match_rate = n_match / max(n_pt, 1)
med = float(np.median(matched_scores)) if matched_scores else 0.0
p95 = float(np.percentile(matched_scores, 95)) if matched_scores else 0.0

consistency = {
    "iou_match": IOU_MATCH,
    "conf": CONF,
    "n_frames": len(CTRL),
    "n_boxes_torch": n_pt,
    "n_boxes_onnx": n_ox,
    "n_matched": n_match,
    "match_rate_vs_torch": round(match_rate, 4),
    "score_abs_diff_median": round(med, 5),
    "score_abs_diff_p95": round(p95, 5),
    "unmatched_torch": unmatched_pt,
    "unmatched_onnx": unmatched_ox,
    "unmatched_near_conf_threshold": near_thr_unmatched,
    "note": "Differences often appear near conf threshold or from NMS numeric order.",
}
print(json.dumps(consistency, indent=2))
(OUT / "consistency.json").write_text(json.dumps(consistency, indent=2))

## 5. Latency protocol

- warmup 10 iterations  
- measure N frames, batch=1  
- sync CUDA if available  
- report p50, p95 ms/frame, throughput FPS, peak GPU mem if CUDA

In [ ]:
def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def bench_predict(predict_fn, paths, warmup=10, tag=""):
    # warmup
    for i in range(min(warmup, len(paths))):
        predict_fn(paths[i % len(paths)])
    sync()
    times = []
    for p in paths:
        sync()
        t0 = time.perf_counter()
        predict_fn(p)
        sync()
        times.append((time.perf_counter() - t0) * 1000.0)
    arr = np.array(times)
    mem = None
    if torch.cuda.is_available():
        mem = round(torch.cuda.max_memory_allocated() / 1e6, 1)
    return {
        "tag": tag,
        "n": len(arr),
        "p50_ms": round(float(np.percentile(arr, 50)), 3),
        "p95_ms": round(float(np.percentile(arr, 95)), 3),
        "mean_ms": round(float(arr.mean()), 3),
        "fps_p50": round(1000.0 / float(np.percentile(arr, 50)), 2),
        "gpu_mem_MB": mem,
    }

def pt_one(p):
    model.predict(str(p), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)

def ox_one(p):
    model_onnx.predict(str(p), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
row_pt = bench_predict(pt_one, BENCH, tag="pytorch_ultralytics")
print("PT", row_pt)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
row_ox = bench_predict(ox_one, BENCH, tag="onnx_ultralytics_ort")
print("ONNX", row_ox)

row_trt = {
    "tag": "tensorrt_fp16",
    "status": "not_built",
    "reason": "No dedicated NVIDIA TRT build session in student Colab path; optional per assignment. Can be built on AWS g4dn / local RTX with NGC container.",
    "p50_ms": None,
    "p95_ms": None,
    "fps_p50": None,
    "gpu_mem_MB": None,
    "engine_build_s": None,
}
print("TRT", row_trt)

table = {
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "imgsz": IMGSZ,
    "batch": 1,
    "warmup": 10,
    "n_frames": len(BENCH),
    "rows": [row_pt, row_ox, row_trt],
}
(OUT / "latency_table.json").write_text(json.dumps(table, indent=2))
print(json.dumps(table, indent=2))

## 6. README commands (copy to repo)

```bash
# Export
python -c "from ultralytics import YOLO; YOLO('yolov8n.pt').export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)"

# Or execute this notebook end-to-end on Colab GPU
```

Artifacts in `/content/pkg09_out/`:
- `yolov8n_imgsz640_opset12.onnx`
- `export_meta.json`
- `consistency.json`
- `latency_table.json`

Paste numbers into `09_Deploy_Package_Serhii_Polishchuk.md`.